# BQP-GUB reference runs (CPLEX + local search) in Jupyter

Run the cells **top to bottom**. The long job runs in a **separate background process**, so you can close or restart
the notebook and re-run the *Monitor* cell later; use *Resume* to continue after an interruption.

Put `bqpgub_run.py` and `bqpgub_parallel.py` in the same folder as this notebook.

## 1. Configuration  (edit this cell)

In [ ]:
from pathlib import Path

DATA_DIR  = Path("data")        # folder with your instance files
PATTERN   = "*"                 # e.g. "*.npz", "*.mat", "*.txt"
OUT       = "reference.csv"     # results file (appended to; safe to resume)
LOG       = "ref.log"

CPLEX_TIME_LIMIT = 1800         # seconds per instance for CPLEX
CPLEX_MAX_N      = 1000         # skip CPLEX above this n (0 = never skip)
CPLEX_THREADS    = 1            # 1 = comparable with the single-thread ADMM codes
LS_TIME_LIMIT    = 300          # cap for the local search (seconds)
SOLVERS          = "cplex,ls"   # reference runs; the ADMM methods come later

## 2. Check the environment

In [ ]:
import sys, os, shutil, importlib
print("python:", sys.executable)
print("CPU cores:", os.cpu_count())
for mod in ["numpy", "scipy", "pandas", "cplex"]:
    try:
        m = importlib.import_module(mod); print(f"{mod:8s} OK  {getattr(m, '__version__', '')}")
    except Exception as e:
        print(f"{mod:8s} MISSING -> run:  %pip install {mod}")
for f in ["bqpgub_run.py", "bqpgub_parallel.py"]:
    print(f, "found" if Path(f).exists() else "NOT FOUND (copy it next to this notebook)")
files = sorted(p for p in DATA_DIR.glob(PATTERN) if p.is_file())
print(len(files), "instance files in", DATA_DIR.resolve())

## 3. Pilot on three files (checks that your file format is read)
Takes about a minute. If a file is not understood you will see a `[skip] ...` line: send me the first 10-15 lines of that file.

In [ ]:
import subprocess, re
def natural(p): return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", p.name)]
files = sorted(files, key=natural)
pilot = [files[0], files[len(files)//2], files[-1]] if len(files) >= 3 else files
print("pilot files:", [p.name for p in pilot])

if Path("pilot.csv").exists(): Path("pilot.csv").unlink()
cmd = [sys.executable, "bqpgub_parallel.py", *map(str, pilot), "--solvers", SOLVERS,
       "--cplex-time-limit", "60", "--cplex-max-n", str(CPLEX_MAX_N), "--time-limit", "60",
       "--cplex-threads", str(CPLEX_THREADS), "--out", "pilot.csv"]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-3000:]); print(res.stderr[-2000:])

## 4. Launch the full reference run in the background
Safe to re-run: `--resume` skips instances already in the results file.

In [ ]:
cmd = [sys.executable, "bqpgub_parallel.py", str(DATA_DIR), "--pattern", PATTERN,
       "--solvers", SOLVERS, "--cplex-time-limit", str(CPLEX_TIME_LIMIT), "--cplex-max-n", str(CPLEX_MAX_N),
       "--cplex-threads", str(CPLEX_THREADS), "--time-limit", str(LS_TIME_LIMIT),
       "--out", OUT, "--resume"]
logf = open(LOG, "a")
proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)
Path("ref.pid").write_text(str(proc.pid))
print("started, PID", proc.pid, "| log:", LOG, "| results:", OUT)

## 5. Monitor  (re-run this cell any time)

In [ ]:
import pandas as pd, time
def running(pid):
    try: os.kill(pid, 0); return True
    except OSError: return False
pid = int(Path("ref.pid").read_text()) if Path("ref.pid").exists() else None
print("background job running:", running(pid) if pid else "no job started")
total = len(files)
if Path(OUT).exists():
    df = pd.read_csv(OUT)
    print(f"done: {len(df)} / {total} instances")
    cols = [c for c in ["name","kind","n","ls","ls_t","cplex","cplex_t","cplex_bound","cplex_status"] if c in df]
    display(df[cols].tail(10))
    if "cplex_status" in df:
        print(df.groupby(["kind"] if "kind" in df else df.index.map(lambda _: "all"))["cplex_status"].value_counts())
print("\n--- log tail ---"); print("".join(open(LOG).readlines()[-8:]) if Path(LOG).exists() else "(no log yet)")

## 6. Stop the background job (optional)

In [ ]:
import signal
if Path("ref.pid").exists():
    try: os.killpg(os.getpgid(int(Path("ref.pid").read_text())), signal.SIGTERM); print("stopped")
    except Exception as e: print("not running:", e)

## 7. Summary table (works on partial results too)
Writes `tables.tex` and prints the gap-to-best-known table.

In [ ]:
res = subprocess.run([sys.executable, "bqpgub_parallel.py", "--summarize", OUT, "--latex", "tables.tex"],
                     capture_output=True, text=True)
print(res.stdout); print(res.stderr[-1500:])

## 8. Send me
`reference.csv` (and any `[skip]` lines from the pilot or `ref.log`). Next step: the paper's own methods
(AADMM / PAL / MADMM-ALU with variable group sizes) run on the same files and reuse these reference values.